# 03 (joint) — Hyperparameter optimisation: WCO (exploration) → MOA (exploitation)
MAS Lab 2 — Fake news detection — **joint text model (FineFake + FakeNewsNet)**

**Input:** `features/` from notebook `02_joint_text_features` (attach ONLY this one). **Accelerator:** GPU T4 x2.

Same code, same search space, same methods as the FineFake notebook 03; only the data changes:
128 text features (mpnet + PCA), ~29,200 train rows from both datasets.
XGBoost hyperparameters are searched on the **train rows only** (fitness = 1 − macro-F1, stratified 3-fold CV).
Methods compared with the **same evaluation budget**: Random search, WCO alone, MOA alone, WCO→MOA (hybrid). 30 independent runs each.
Every run is saved as soon as it finishes (`results/`). A **time guard** stops starting new runs before Kaggle's 12 h limit,
so the commit always finishes and its output is kept.

## 0. Setup: GPU, input features

In [1]:
import os, sys, glob, json, time
import numpy as np
import pandas as pd
import torch

SEED = 42
N_GPU = torch.cuda.device_count()
print('GPUs:', N_GPU, [torch.cuda.get_device_name(i) for i in range(N_GPU)])

found = glob.glob('/kaggle/input/**/X_text_pca.npy', recursive=True)
assert found, 'features not found: attach the 02_joint_text_features output (Add Input)'
assert len(found) == 1, f'several feature folders attached, keep ONLY 02_joint_text_features: {found}'
FEAT = os.path.dirname(found[0])
print('Features folder:', FEAT)
assert not os.path.exists(f'{FEAT}/X_image_pca.npy'), 'this is the FineFake (02b) output, attach 02_joint_text_features instead'

X     = np.load(f'{FEAT}/X_text_pca.npy').astype(np.float32)
y     = np.load(f'{FEAT}/y.npy').astype(int)
is_tr = np.load(f'{FEAT}/is_train.npy')
meta  = pd.read_pickle(f'{FEAT}/meta.pkl')

X_tr, y_tr = X[is_tr], y[is_tr]
print('X:', X.shape, '| train:', X_tr.shape, '| fake/real in train:', np.bincount(y_tr))
print(pd.crosstab(meta.loc[is_tr, 'dataset'], meta.loc[is_tr, 'label'].map({0: 'fake', 1: 'real'})))

GPUs: 2 ['Tesla T4', 'Tesla T4']
Features folder: /kaggle/input/notebooks/khitersaranadjet/02-joint-text-features/features
X: (36516, 128) | train: (29212, 128) | fake/real in train: [10486 18726]
label        fake   real
dataset                 
fakenewsnet  4092  12882
finefake     6394   5844


## 1. Metaheuristic code (same files as `src/metaheuristics/` in the repo)
Written to disk so the parallel workers can import them.

In [2]:
os.makedirs('src/metaheuristics', exist_ok=True)
open('src/__init__.py', 'w').close()
open('src/metaheuristics/__init__.py', 'w').close()

In [3]:
%%writefile src/metaheuristics/common.py
"""Shared tools for all metaheuristics: a budget-limited objective that records the search history."""
import numpy as np

class BudgetExhausted(Exception):
    """Raised when the evaluation budget is used up (stops any algorithm cleanly)."""

class Objective:
    """Wraps a fitness function f(x), x in [0,1]^D, minimised.
    - counts evaluations and stops at `budget`
    - remembers the best solution found so far
    - records the history needed for the convergence curves
    """

    def __init__(self, func, budget):
        self.func = func
        self.budget = budget
        self.n_evals = 0
        self.best_f = np.inf
        self.best_x = None
        self.best_history = []
        self.values = []
        self.phase = []
        self.current_phase = ''

    @property
    def left(self):
        return self.budget - self.n_evals

    def __call__(self, x):
        if self.n_evals >= self.budget:
            raise BudgetExhausted
        x = np.clip(np.asarray(x, dtype=float), 0.0, 1.0)
        f = float(self.func(x))
        self.n_evals += 1
        if f < self.best_f:
            self.best_f, self.best_x = f, x.copy()
        self.best_history.append(self.best_f)
        self.values.append(f)
        self.phase.append(self.current_phase)
        return f

def evaluate_all(obj, X):
    """Evaluate the rows of X until the budget runs out; returns the fitness of the rows evaluated."""
    fit = []
    for x in X:
        if obj.left == 0:
            break
        fit.append(obj(x))
    return np.array(fit)

Writing src/metaheuristics/common.py


In [4]:
%%writefile src/metaheuristics/wco.py
"""World Cup Optimization (WCO)
Used here as the EXPLORATION phase.

Mapping of the paper to this implementation (interpretation documented in the report):
  team                 -> one candidate solution x in [0,1]^D
  continents           -> groups of teams; each starts in its own region with its own spread
  continent score      -> Rank = (beta * std + mean) / 2 of the members' fitness   (paper eq. 10)
  preliminary round    -> the 2 best teams of each continent qualify (X_Rank)       (eq. 13)
  champion             -> best team of the cup                                       (eq. 14)
  next cup population  -> Pop = [X_Best, X_Rand] split by the cross point CP        (eq. 15, 19)
       X_Rand : new random teams anywhere in the bounds        (exploration)
       X_Best : teams generated near qualified teams, within a radius set by the accuracy
                coefficient ac; better-ranked continents produce more of them   (eq. 16-18)
  the champion is kept (elitism) so the best solution is never lost.
"""
import numpy as np
from .common import BudgetExhausted, evaluate_all

def wco(obj, dim, rng, pop_size=15, n_continents=5, beta=0.5, cp=0.5, ac=0.2):
    """Run WCO until obj's budget is used. Returns the final population (X, fit), sorted best first."""
    obj.current_phase = 'WCO'
    sizes = np.full(n_continents, pop_size // n_continents)
    sizes[: pop_size % n_continents] += 1
    labels = np.repeat(np.arange(n_continents), sizes)

    spreads = np.linspace(0.10, 0.40, n_continents)
    X = np.vstack([np.clip(rng.normal(rng.random(dim), spreads[c], (sizes[c], dim)), 0, 1)
                   for c in range(n_continents)])

    try:
        fit = evaluate_all(obj, X)
        if len(fit) < len(X):
            raise BudgetExhausted
        while obj.left > 0:
            scores = np.array([(beta * fit[labels == c].std(ddof=1) + fit[labels == c].mean()) / 2
                               for c in range(n_continents)])
            order = np.argsort(scores)

            qualified, weights = [], []
            for rank_pos, c in enumerate(order):
                members = np.where(labels == c)[0]
                top2 = members[np.argsort(fit[members])[:2]]
                qualified.extend(top2)
                weights.extend([n_continents - rank_pos] * len(top2))
            qualified = np.array(qualified)
            weights = np.array(weights, dtype=float) / np.sum(weights)
            champ = int(np.argmin(fit))

            n_new = pop_size - 1
            n_rand = int(round(cp * n_new))
            n_best = n_new - n_rand
            parents = rng.choice(qualified, size=n_best, p=weights)
            radius = ac / 2
            X_best = np.clip(X[parents] + rng.uniform(-radius, radius, (n_best, dim)), 0, 1)
            X_rand = rng.random((n_rand, dim))
            X_new = np.vstack([X_best, X_rand])

            fit_new = evaluate_all(obj, X_new)
            X = np.vstack([X[champ], X_new[: len(fit_new)]])
            fit = np.concatenate([[fit[champ]], fit_new])
            if len(fit) < pop_size:
                break

    except BudgetExhausted:
        pass

    order = np.argsort(fit)
    return X[: len(fit)][order], fit[order]

Writing src/metaheuristics/wco.py


In [5]:
%%writefile src/metaheuristics/moa.py
"""Morabaraba Optimization Algorithm (MOA)
Used here as the EXPLOITATION phase.

Game -> optimisation mapping (paper section 3):
  cow (agent)            -> one candidate solution x
  two teams Ta / Tb      -> population split in two competing halves
  cow count c_i          -> per-agent state (starts at 12) that selects the phase: (energy)
                              Placing  c >= 9      : Levy pull toward the team leader        (eq. 4)
                              Moving   3 < c < 9   : mill check + 3 candidate moves          (eq. 5-9)
                              Flying   c == 3      : Levy jump around the global best        (eq. 10)
                              c <= 2               : cow respawned at a random position
  mill                   -> 3 teammates aligned on a board line (a triple of dimensions)   (eq. 5)
  cow shooting           -> after a mill, the worst opposing cow loses one cow and is
                            dragged toward the mill on its worst-aligned dimension       (eq. 6)
  mill roster            -> the K last mills, used as a multi-leader pool (roulette)     (eq. 12)
  mood, velocity, retreat-> x += eta*v - zeta*sign(m)*(x_worst - x)                       (eq. 11, 13)

Implementation choices
  - velocity / retreat are applied to each proposal BEFORE it is evaluated (no extra evaluation)
  - the mill check runs for every cow after its move (flowchart), at most one mill per cow per turn
  - a mill's roster entry is the position of the best cow of the triple
  - teams are formed by alternating the ranked population (both teams get strong cows)
"""
import numpy as np
from .common import BudgetExhausted, evaluate_all

def _board_mills():
    mills = []
    for s in range(3):
        b = 8 * s
        mills += [(b + 0, b + 1, b + 2), (b + 2, b + 3, b + 4), (b + 4, b + 5, b + 6), (b + 6, b + 7, b + 0)]
    for p in (1, 3, 5, 7):
        mills.append((p, 8 + p, 16 + p))
    for p in (0, 2, 4, 6):
        mills.append((p, 8 + p, 16 + p))
    return mills

def board_lines(dim):
    """Map the 20 board mills to triples of distinct decision variables (round-robin point -> dimension)."""
    lines = []
    for m in _board_mills():
        dims = tuple(sorted({p % dim for p in m}))
        if len(dims) == 3 and dims not in lines:
            lines.append(dims)
    return lines

def levy(rng, size, beta=1.5):
    """Levy-flight steps (Mantegna's algorithm): mostly small steps, occasionally very long jumps."""
    from math import gamma, sin, pi
    sigma = (gamma(1 + beta) * sin(pi * beta / 2) / (gamma((1 + beta) / 2) * beta * 2 ** ((beta - 1) / 2))) ** (1 / beta)
    u = rng.normal(0, sigma, size)
    v = rng.normal(0, 1, size)
    return u / np.abs(v) ** (1 / beta)

def tent_obl_init(obj, n, dim, rng, mu=1.999):
    """Chaotic Tent-map initialisation + Opposition-Based Learning: evaluate 2n points, keep the best n."""
    z = rng.random(dim)
    X = np.empty((n, dim))
    for i in range(n):
        z = np.where(z < 0.5, mu * z, mu * (1 - z))
        z = np.clip(z, 1e-6, 1 - 1e-6)
        X[i] = z
    X_all = np.vstack([X, 1.0 - X])
    fit = evaluate_all(obj, X_all)
    X_all = X_all[: len(fit)]
    order = np.argsort(fit)[:n]
    return X_all[order], fit[order]

def moa(obj, dim, rng, pop_size=10, init_X=None, init_fit=None,
        alpha=0.05, tau=0.05, gamma_drag=0.30, F=0.8, b=1.0, eta=0.40, zeta=0.20, K=5, c0=12):

    """Run MOA until obj's budget is used. Optional init_X/init_fit = seed population (e.g. from WCO).
    Returns the final population (X, fit) sorted best first, and phase statistics."""
    obj.current_phase = 'MOA'
    stats = {'placing': 0, 'moving': 0, 'flying': 0, 'mills': 0, 'respawns': 0}
    try:
        if init_X is None:
            X, fit = tent_obl_init(obj, pop_size, dim, rng)
        else:
            X, fit = np.array(init_X[:pop_size], dtype=float), np.array(init_fit[:pop_size], dtype=float)
        n = len(X)
        order = np.argsort(fit)
        X, fit = X[order], fit[order]
        team = np.arange(n) % 2
        cows = np.full(n, c0)
        vel = np.zeros((n, dim))
        roster = []
        lines = board_lines(dim)

        def propose(i, x_new, m):
            """Velocity persistence + retreat from the worst teammate, then stay in bounds (eq. 13)."""
            mates = np.where(team == team[i])[0]
            x_worst = X[mates[np.argmax(fit[mates])]]
            return np.clip(x_new + eta * vel[i] - zeta * np.sign(m) * (x_worst - X[i]), 0, 1)

        while obj.left > 0:
            for i in range(n):
                if obj.left == 0:
                    break
                mates = np.where(team == team[i])[0]
                lead = X[mates[np.argmin(fit[mates])]]
                m = 2 * rng.random() - 1
                c = cows[i]

                if c <= 2:
                    stats['respawns'] += 1
                    x_new = rng.random(dim)
                    f_new = obj(x_new)
                    cows[i] = c0
                elif c >= 9:
                    stats['placing'] += 1
                    x_new = propose(i, lead + alpha * levy(rng, dim) * (lead - X[i]), m)
                    f_new = obj(x_new)
                elif c == 3:
                    stats['flying'] += 1
                    best = obj.best_x
                    x_new = propose(i, best + rng.normal(0, 1, dim) * levy(rng, dim) * np.abs(best - X[i]), m)
                    f_new = obj(x_new)
                else:
                    stats['moving'] += 1
                    if roster:
                        q = np.array([r[1] for r in roster]); q = q / q.sum()
                        lead_m = roster[rng.choice(len(roster), p=q)][0]
                    else:
                        lead_m = lead
                    r1, r2 = rng.choice(mates, 2, replace=len(mates) < 2)
                    l = rng.uniform(-1, 1)
                    A = X[i] + rng.random() * (lead_m - X[i])
                    B = lead_m + F * (X[r1] - X[r2])
                    C = np.abs(lead_m - X[i]) * np.exp(b * l) * np.cos(2 * np.pi * l) + lead_m
                    cands = [propose(i, z, m) for z in (A, B, C)]
                    f_c = [obj(z) for z in cands]
                    k = int(np.argmin(f_c))
                    x_new, f_new = cands[k], f_c[k]

                if f_new < fit[i]:
                    d = x_new - X[i]
                    vel[i] = d / (np.linalg.norm(d) + 1e-12)
                X[i], fit[i] = x_new, f_new

                mates = np.where(team == team[i])[0]
                others = [j for j in mates[np.argsort(fit[mates])] if j != i][:2]
                if len(others) == 2:
                    trio = X[[i] + others]
                    for L in lines:
                        vals = trio[:, L]
                        spread = vals.max(axis=0) - vals.min(axis=0)
                        Q = 1 - np.max(spread / np.maximum(np.abs(vals).max(axis=0), 1e-12))
                        if Q >= 1 - tau:
                            stats['mills'] += 1
                            opp = np.where(team != team[i])[0]
                            w = opp[np.argmax(fit[opp])]
                            cows[w] -= 1
                            mu_L = vals.mean(axis=0)
                            k = int(np.argmax(np.abs(X[w, list(L)] - mu_L)))
                            X[w, L[k]] = (1 - gamma_drag) * X[w, L[k]] + gamma_drag * mu_L[k]
                            roster.append((trio[np.argmin(fit[[i] + others])].copy(), Q))
                            roster = roster[-K:]
                            break
    except BudgetExhausted:
        pass

    order = np.argsort(fit)
    return X[order], fit[order], stats

Writing src/metaheuristics/moa.py


In [6]:
%%writefile src/metaheuristics/hybrid.py
"""Hybrid WCO -> MOA: WCO explores for the first part of the budget, then MOA exploits,
starting from WCO's best solutions."""
from .wco import wco
from .moa import moa

def wco_moa(obj, dim, rng, switch=0.5, wco_kwargs=None, moa_kwargs=None):
    """Spend `switch` x budget on WCO, the rest on MOA seeded with WCO's best agents."""
    total = obj.budget
    obj.budget = int(round(switch * total))
    X, fit = wco(obj, dim, rng, **(wco_kwargs or {}))
    switch_eval = obj.n_evals
    obj.budget = total
    X, fit, stats = moa(obj, dim, rng, init_X=X, init_fit=fit, **(moa_kwargs or {}))
    stats['switch_eval'] = switch_eval
    return X, fit, stats

Writing src/metaheuristics/hybrid.py


## 2. Search space, decoding and fitness
An agent is a vector in [0,1]^9, decoded into XGBoost hyperparameters.
The 3 CV folds are fixed once (same seed) so every method is scored on exactly the same splits.

In [7]:
%%writefile src/fitness.py
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

SPACE = {
    'n_estimators':     (100, 600, 'lin', True),
    'learning_rate':    (0.01, 0.3, 'log', False),
    'max_depth':        (3, 10, 'lin', True),
    'min_child_weight': (1, 10, 'lin', False),
    'subsample':        (0.5, 1.0, 'lin', False),
    'colsample_bytree': (0.3, 1.0, 'lin', False),
    'gamma':            (0, 5, 'lin', False),
    'reg_alpha':        (0, 5, 'lin', False),
    'reg_lambda':       (0, 5, 'lin', False),
}
DIM = len(SPACE)

def decode(x):
    """[0,1]^9 vector -> dict of XGBoost hyperparameters."""
    params = {}
    for v, (name, (lo, hi, scale, is_int)) in zip(np.clip(x, 0, 1), SPACE.items()):
        val = 10 ** (np.log10(lo) + v * (np.log10(hi) - np.log10(lo))) if scale == 'log' else lo + v * (hi - lo)
        params[name] = int(round(val)) if is_int else float(val)
    return params

def make_model(params, device='cuda', seed=42):
    return XGBClassifier(**params, tree_method='hist', device=device, eval_metric='logloss',
                         random_state=seed, n_jobs=4, verbosity=0)

def make_fitness(X, y, device='cuda', n_splits=3, seed=42):
    """Returns f(x) = 1 - mean macro-F1 over fixed stratified folds (lower is better)."""
    folds = list(StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed).split(X, y))
    def fitness(x):
        scores = []
        for tr, va in folds:
            model = make_model(decode(x), device, seed)
            model.fit(X[tr], y[tr])
            scores.append(f1_score(y[va], model.predict(X[va]), average='macro'))
        return 1.0 - float(np.mean(scores))
    return fitness

Writing src/fitness.py


In [8]:
sys.path.insert(0, os.getcwd())
from src.fitness import SPACE, DIM, decode, make_fitness
from src.metaheuristics.common import Objective
from src.metaheuristics.wco import wco
from src.metaheuristics.moa import moa
from src.metaheuristics.hybrid import wco_moa

print('Example decoding of x = 0.5 everywhere:')
print(decode(np.full(DIM, 0.5)))

Example decoding of x = 0.5 everywhere:
{'n_estimators': 350, 'learning_rate': 0.05477225575051662, 'max_depth': 6, 'min_child_weight': 5.5, 'subsample': 0.75, 'colsample_bytree': 0.6499999999999999, 'gamma': 2.5, 'reg_alpha': 2.5, 'reg_lambda': 2.5}


## 3. Timed test
Score XGBoost's default settings, then time a few random settings to size the budget.

In [9]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from xgboost import XGBClassifier

dev = 'cuda' if N_GPU else 'cpu'
t = time.time()
cv = StratifiedKFold(3, shuffle=True, random_state=SEED)
f1_default = cross_val_score(XGBClassifier(tree_method='hist', device=dev, random_state=SEED, verbosity=0),
                             X_tr, y_tr, cv=cv, scoring='f1_macro').mean()
print(f'Default XGBoost: CV macro-F1 = {f1_default:.4f}  (fitness {1 - f1_default:.4f})  [{time.time() - t:.1f}s]')

fit_fn = make_fitness(X_tr, y_tr, device=dev, seed=SEED)
rng = np.random.default_rng(0)
times = []
for _ in range(6):
    x = rng.random(DIM)
    t = time.time(); f = fit_fn(x); times.append(time.time() - t)
    print(f'fitness {f:.4f}  time {times[-1]:5.1f}s  {decode(x)}')
SEC_PER_EVAL = float(np.mean(times[1:]))
print(f'\nMean time per evaluation (3 folds): {SEC_PER_EVAL:.2f}s')

Default XGBoost: CV macro-F1 = 0.7063  (fitness 0.2937)  [2.6s]
fitness 0.3264  time   2.3s  {'n_estimators': 418, 'learning_rate': 0.025032699681422646, 'max_depth': 3, 'min_child_weight': 1.1487487197567618, 'subsample': 0.9066351196001362, 'colsample_bytree': 0.9389289040944051, 'gamma': 3.0331788788358995, 'reg_alpha': 3.647482804919992, 'reg_lambda': 2.7181249573271145}
fitness 0.2878  time   2.8s  {'n_estimators': 568, 'learning_rate': 0.16036685991738303, 'max_depth': 3, 'min_child_weight': 8.716638489288124, 'subsample': 0.5167927876527322, 'colsample_bytree': 0.8107588125009608, 'gamma': 0.8782781030127951, 'reg_alpha': 4.315894611749433, 'reg_lambda': 2.7073061012454587}
fitness 0.3259  time   1.7s  {'n_estimators': 250, 'learning_rate': 0.04210757962128049, 'max_depth': 3, 'min_child_weight': 2.1185494884960754, 'subsample': 0.8353122073468151, 'colsample_bytree': 0.7530326581019751, 'gamma': 3.0769255574062693, 'reg_alpha': 1.9183877713094173, 'reg_lambda': 4.98604967894605

## 4. Budget
Total evaluations = methods × runs × budget. Runs are spread over the available GPUs.
The budget is chosen **automatically** from the timed test: 80 (same as FineFake) if it fits in ~8.5 h (with the 1.5× slowdown measured on FineFake), otherwise
the largest multiple of 10 that fits. Every method always gets the same budget.

In [10]:
METHODS = ['random', 'wco', 'moa', 'wco_moa']
N_RUNS  = 30
SWITCH  = 0.5
TARGET_H = 8.5
MAX_BUDGET = 80
SLOWDOWN = 1.5

fits = TARGET_H * 3600 * max(N_GPU, 1) / (len(METHODS) * N_RUNS * SEC_PER_EVAL * SLOWDOWN)
BUDGET = int(max(40, min(MAX_BUDGET, fits // 10 * 10)))

total_evals = len(METHODS) * N_RUNS * BUDGET
est_h = total_evals * SEC_PER_EVAL * SLOWDOWN / max(N_GPU, 1) / 3600
print(f'BUDGET = {BUDGET} evaluations per run')
print(f'{total_evals} evaluations -> estimated {est_h:.1f} h on {max(N_GPU, 1)} GPU(s) (including the x{SLOWDOWN} slowdown)')

BUDGET = 70 evaluations per run
8400 evaluations -> estimated 8.5 h on 2 GPU(s) (including the x1.5 slowdown)


## 5. Runs (parallel over GPUs, each run saved immediately)
Run index is the outer loop, so if time runs out every method still has (almost) the same number of finished runs.
**Time guard:** before each new pair of runs, if the slowest pair so far would not finish before `STOP_AFTER_H`,
the loop stops. Notebook 04 only keeps runs finished by all 4 methods (paired comparison).

In [11]:
from joblib import Parallel, delayed
OUT = '/kaggle/working/results'
os.makedirs(OUT, exist_ok=True)

def run_one(method, run, gpu, X_tr, y_tr, budget, switch, seed):
    import sys, os, json, time
    import numpy as np
    sys.path.insert(0, '/kaggle/working')
    from src.fitness import DIM, decode, make_fitness
    from src.metaheuristics.common import Objective
    from src.metaheuristics.wco import wco
    from src.metaheuristics.moa import moa
    from src.metaheuristics.hybrid import wco_moa

    path = f'/kaggle/working/results/{method}_run{run:02d}.json'
    if os.path.exists(path):
        return path

    device = f'cuda:{gpu}' if gpu is not None else 'cpu'
    obj = Objective(make_fitness(X_tr, y_tr, device=device, seed=seed), budget)
    rng = np.random.default_rng(1000 + run)
    t0 = time.time(); stats = {}

    if method == 'random':
        obj.current_phase = 'RANDOM'
        for _ in range(budget):
            obj(rng.random(DIM))

    elif method == 'wco':
        wco(obj, DIM, rng)

    elif method == 'moa':
        _, _, stats = moa(obj, DIM, rng)

    elif method == 'wco_moa':
        _, _, stats = wco_moa(obj, DIM, rng, switch=switch)

    res = {'method': method, 'run': run, 'best_f': obj.best_f, 'best_x': obj.best_x.tolist(),
           'best_params': decode(obj.best_x), 'best_history': obj.best_history, 'values': obj.values,
           'phase': obj.phase, 'stats': {k: int(v) for k, v in stats.items()}, 'seconds': time.time() - t0}

    with open(path, 'w') as f:
        json.dump(res, f)
    return path

STOP_AFTER_H = 11.0
jobs = [(m, r) for r in range(N_RUNS) for m in METHODS]
gpus = list(range(N_GPU)) or [None]
t_all = time.time(); slowest = 0.0
for start in range(0, len(jobs), len(gpus)):
    elapsed = time.time() - t_all
    if elapsed + slowest > STOP_AFTER_H * 3600:
        print(f'Time guard: stopping after {elapsed / 3600:.1f} h ({start}/{len(jobs)} runs done)')
        break
    batch = jobs[start:start + len(gpus)]
    t_b = time.time()
    Parallel(n_jobs=len(batch))(delayed(run_one)(m, r, gpus[k], X_tr, y_tr, BUDGET, SWITCH, SEED)
                                for k, (m, r) in enumerate(batch))
    slowest = max(slowest, time.time() - t_b)
    done = start + len(batch)
    print(f'{done}/{len(jobs)} runs done | elapsed {(time.time() - t_all) / 60:.1f} min | last: {batch}')

2/120 runs done | elapsed 6.6 min | last: [('random', 0), ('wco', 0)]
4/120 runs done | elapsed 17.9 min | last: [('moa', 0), ('wco_moa', 0)]
6/120 runs done | elapsed 23.8 min | last: [('random', 1), ('wco', 1)]
8/120 runs done | elapsed 31.7 min | last: [('moa', 1), ('wco_moa', 1)]
10/120 runs done | elapsed 37.9 min | last: [('random', 2), ('wco', 2)]
12/120 runs done | elapsed 46.4 min | last: [('moa', 2), ('wco_moa', 2)]
14/120 runs done | elapsed 53.3 min | last: [('random', 3), ('wco', 3)]
16/120 runs done | elapsed 60.3 min | last: [('moa', 3), ('wco_moa', 3)]
18/120 runs done | elapsed 66.6 min | last: [('random', 4), ('wco', 4)]
20/120 runs done | elapsed 75.9 min | last: [('moa', 4), ('wco_moa', 4)]
22/120 runs done | elapsed 82.4 min | last: [('random', 5), ('wco', 5)]
24/120 runs done | elapsed 90.4 min | last: [('moa', 5), ('wco_moa', 5)]
26/120 runs done | elapsed 95.6 min | last: [('random', 6), ('wco', 6)]
28/120 runs done | elapsed 104.4 min | last: [('moa', 6), ('wco

## 6. Quick summary (full analysis in notebook 04)

In [12]:
import pandas as pd
rows = [json.load(open(p)) for p in sorted(glob.glob(f'{OUT}/*.json'))]
df = pd.DataFrame([{'method': r['method'], 'run': r['run'], 'best_f': r['best_f'],
                    'cv_f1': 1 - r['best_f'], 'seconds': r['seconds']} for r in rows])
print(df.groupby('method')[['cv_f1', 'seconds']].agg(['mean', 'std', 'count']).round(4))
print(f'\nDefault XGBoost CV macro-F1: {f1_default:.4f}')
df.to_csv(f'{OUT}/summary.csv', index=False)
print(f'Budget used: {BUDGET} evaluations per run')

          cv_f1                 seconds                
           mean     std count      mean       std count
method                                                 
moa      0.7304  0.0027    30  571.1013  161.4658    30
random   0.7289  0.0017    30  339.3400   23.1146    30
wco      0.7290  0.0021    30  373.8500   55.2102    30
wco_moa  0.7296  0.0030    30  504.6499  133.2816    30

Default XGBoost CV macro-F1: 0.7063
Budget used: 70 evaluations per run
